In [6]:
%load_ext autoreload 
%autoreload 2

In [7]:
from pomap_v2 import pomap
import polars as pl
import datetime as dt

import numpy as np

In [11]:
example_df = pl.from_dict(
    {'cat1': ['a',  'b',],}
)

timestamps = pl.datetime_range(dt.datetime(2025, 1, 1),
                               dt.datetime(2025, 1, 7),
                               interval='1h',
                               eager=True
                               ).rename('ts').to_frame()

example_df = example_df.join(timestamps, how='cross')
example_df = example_df.with_columns(
    feature = np.random.normal(0, 1, example_df.shape[0])
)

example_df = example_df.with_columns(hour=pl.col('ts').dt.hour())

### Before we get started, let's prototype the functional behaviour we want on a simpler problem

In [41]:
from typing import Self

class Operation:
    def __init__(self, nodes: list[Self]):
        self.nodes = nodes

    def operate(self, x):
        current = x
        for node in self.nodes:
            current = node.operate(current)
        return current

    def compose(self, other: Self):
        return Operation(self.nodes + other.nodes)

    def __sum__(self, other: Self):
        return self.compose(other)

class PrimitiveOperation(Operation):

    def __init__(self):
        self.nodes = [self]


class Add2(PrimitiveOperation):

    def operate(self, x):
        return x + 2


In [42]:
add2 = Add2()
add4 = add2.compose(add2)

In [40]:
add4.operate(1)

5

### Let's start prototyping a new PoMap

In [19]:
# First, a sketch, a Pomap is defined by a 'dimension' and a set of expected labels     

class Pomap:

    # A PoMap is defined by a 'dimension' and a set of labels belonging to that dimension
    def __init__(self, nodes: list[Self], name: str):

        self.name = name
        self._nodes = nodes

        # Implement some standardised 
        self._train_column_name = f'train({self.name})'
        self._test_column_name = f'test({self.name})'
        self._validate_column_name = f'validate({self.name})'

    def __repr__(self):
        return self.name

    # These three (train, test, validate) functions define the behaviour of the PoMap.
    # E.g, is it a cross validation, is it categorical, etc.
    # See below for an example of a reasonably complex example.
    def label_rows_as_train(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

    # There has to be a separate one for test and validation, because
    # train and test data must be distinct.
    def label_rows_as_test(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

    # .... as above
    def label_rows_as_validate(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

    # Every PoMap function must specify a way of representing itself as a DataFrame
    # From which we can 'read off' the label information
    @property
    def labels_df(self) -> pl.Series:
        raise NotImplementedError

    def __getitem__(self, dimension):
        return 

# -=-=-=-=-=-=-=-==-=-=-==-=--=-==-=-=--=-=-=-=-=-=-=-=-=-=-=-=-=-==--=-==-=--=-=-=
    # We also add some helper functions to make sure we have consistent formatting
    # Everywhere

    def train_column_name(self): return f'train({self.name})'

# -=-=-=-=-=-=-=--=-=-==-=-=-=-=-=--=-=-=-=-=-=-=-=-=-=-=-=-==-=-=-=-=-=-=-=-=-=--=-
#   After this, things get interesting, since we start to deal with how the pomap actually behaves

# This is simple enough, you use the label to train function, filter, and drop the metadata 
    def label_to_train(self, df: pl.DataFrame, label: dict):
        # Adds a column called is_train
        df = self.label_rows_as_train(df, label)
        df = df.filter(self.train_column_name())
        df = df.drop('is_train')

        return df

    def label_rows_as_train(self, df: pl.DataFrame, label):
        df = df.with_columns(**{f'is_train_{node.name}': }

        # Pack everything into a 
        df = df.

    def compose(self, other: Self):
        return Pomap(nodes=self._nodes + other._nodes, name=f'{self.name} + {other.name}')

    def labels(self, other: Self):
        

In [37]:
# An example of a child class.

class RandomisedCrossValidationPoMap(Pomap):

    def __init__(self, dims, folds: int, index_column: 'str'):
        super().__init__(nodes=[self], name='RandomisedCrossValidation')
        self.fold_labels = [str(c) for c in range(folds)]
    
    def label_rows_as_test(self, df: pl.DataFrame, label: dict):
        assert df.unique(self.index_column).shape[0] == df.shape[0]
        index_to_test_fold = {}
        for index in df.select(self.index_column).unique():
            index_to_test_fold[index] = np.random.choice(self.fold_labels)

        df = df.with_columns(
            pl.col(self.index_column)
            .replace_strict(index_to_test_fold)
            .alias(self.train_column_name(label, self.dims))
        )

        return df

    def label_rows_as_train(self, df, label):
        df = self.label_rows_as_test(df, label)
        test_column = self.test_column_name(label, self.dims)

        # A node is in test if it's label did NOT appear in the train set.
        df = df.with_columns(
            ~(pl.col(test_column))
            .alias(self.train_column_name(label, self.dims))
        )
        df = df.drop(test_column)

        return df
        

### A little aside, can I do up a simple pathwise in Python

5